# Impact of model size (capacity)

Skill vs number of trainable parameters (stencil 3x3). Widths [16,16]->[64,64] carry seed error bars (seeds 0-2); depth variants [32], [32,32,32] shown too. Part 1 found skill most sensitive to parameter count, with a plateau.

In [ ]:
import sys
sys.path.append('../../src/training-on-CM2.6/scripts')
import numpy as np
import matplotlib.pyplot as plt
from sensitivity_eval import depth_mean, depth_profile, FACTORS, SPACING
from sensitivity_configs import cfg

SPAC = [SPACING[f] for f in FACTORS]
def nparams(stencil, hidden):
    sizes = [stencil**2 * 5, *hidden, 2]
    return sum(sizes[i]*sizes[i+1] + sizes[i+1] for i in range(len(sizes)-1))

In [ ]:
widths = ['[16,16]', '[32,32]', '[48,48]', '[64,64]']
def overall(name):
    dm = depth_mean(name, 'R2F')
    return np.mean([dm[f] for f in FACTORS]) if dm else np.nan

fig, ax = plt.subplots(figsize=(6.5, 4.5))
xp, yp, ye = [], [], []
for w in widths:
    vals = [overall(cfg(3, w, s)['name']) for s in (0, 1, 2)]
    vals = [v for v in vals if np.isfinite(v)]
    xp.append(nparams(3, eval(w))); yp.append(np.mean(vals)); ye.append(np.std(vals))
ax.errorbar(xp, yp, yerr=ye, fmt='-o', capsize=3, label='width [w,w] (+/- seed std)')
for hid_str, lab in [('[32]', '1 layer [32]'), ('[32,32,32]', '3 layers')]:
    ax.plot(nparams(3, eval(hid_str)), overall(cfg(3, hid_str, 0)['name']), 's', label=lab)
ax.set_xscale('log'); ax.set_xlabel('# trainable parameters'); ax.set_ylabel('$R^2$ (mean over resolution)')
ax.set_title('Skill vs model size (stencil 3x3)'); ax.legend(); ax.grid(alpha=0.3)
plt.savefig('impact_model_sizes.pdf', bbox_inches='tight'); plt.show()

### $R^2$ vs resolution, by width

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for w in widths:
    dm = depth_mean(cfg(3, w, 0)['name'], 'R2F')
    if dm: ax.plot(SPAC, [dm[f] for f in FACTORS], '-o', label=w)
ax.set_xlabel('coarse-grid spacing [deg]'); ax.set_ylabel('$R^2$ (depth-mean)')
ax.legend(title='hidden'); ax.grid(alpha=0.3); ax.set_title('Skill vs resolution, by width')
plt.savefig('impact_model_sizes_resolution.pdf', bbox_inches='tight'); plt.show()